In [62]:
import pandas as pd
import numpy as np
import pyreadstat
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# ── Machine Learning ──────────────────────────────────────
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from sklearn.tree            import DecisionTreeClassifier
from sklearn.svm             import SVC
from sklearn.neighbors       import KNeighborsClassifier
from xgboost                 import XGBClassifier

# ── Préparation et validation ─────────────────────────────
from sklearn.model_selection import (train_test_split, cross_val_score,
                                      StratifiedKFold, learning_curve)
from sklearn.preprocessing   import StandardScaler

# ── Métriques d'évaluation ────────────────────────────────
from sklearn.metrics import (accuracy_score, roc_auc_score, roc_curve,
                              confusion_matrix, classification_report,
                              f1_score, precision_score, recall_score)

# ── SMOTE (correction déséquilibre) ──────────────────────
from imblearn.over_sampling import SMOTE

# ── Dossiers de sortie ────────────────────────────────────
os.makedirs("../outputs/figures", exist_ok=True)
os.makedirs("../outputs/tables",  exist_ok=True)

# ── Style graphiques ──────────────────────────────────────
plt.rcParams['figure.dpi']        = 130
plt.rcParams['font.family']       = 'DejaVu Sans'
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
sns.set_theme(style='whitegrid')

# ── Couleurs ──────────────────────────────────────────────
NAVY  = "#0D1B4B"
BLUE  = "#2196F3"
TEAL  = "#0097A7"
GREEN = "#388E3C"
CORAL = "#FF5722"
AMBER = "#F57C00"
GRAY  = "#90A4AE"
COLORS_6 = [NAVY, TEAL, CORAL, AMBER, GREEN, BLUE]

print("✓ Toutes les bibliothèques ML chargées !")

✓ Toutes les bibliothèques ML chargées !


In [63]:
# ── Chargement du fichier SAV ────────────────────────────
df_raw, meta = pyreadstat.read_sav("../data/CMIR71FL.SAV")
df_raw.columns = df_raw.columns.str.lower()

# ── Sélection des variables ───────────────────────────────
variables_dhs = {
    'v012': 'age', 'v106': 'niveau_instruction',
    'v201': 'nb_enfants_total', 'v218': 'nb_enfants_vivants',
    'v313': 'utilisation_contraceptif', 'v501': 'statut_matrimonial',
    'v025': 'milieu_residence', 'v024': 'region', 'v130': 'religion',
    'v714': 'travail', 'v190': 'quintile_richesse', 'v602': 'desir_enfant',
}
cols_ok = [c for c in variables_dhs if c in df_raw.columns]
df = df_raw[cols_ok].rename(columns=variables_dhs).copy()

# ── Variable cible ────────────────────────────────────────
df['desir_bin'] = df['desir_enfant'].apply(
    lambda x: 1 if x in [1,2,3] else (0 if x in [4,5,6,7,8] else np.nan))
df = df[df['desir_bin'].notna()].copy()
df['desir_bin'] = df['desir_bin'].astype(int)

# ── Variables dérivées ────────────────────────────────────
#  Codes DHS Cameroun 2018 vérifiés
df['contraceptif_bin']  = (df['utilisation_contraceptif'] > 0).astype(int)
df['region_nord']       = df['region'].isin([1, 4, 6]).astype(int)  # Adamaoua + Extr-Nord + Nord
df['religion_musulman'] = (df['religion'] == 4).astype(int)          # code 4 = Musulman

N     = len(df)
N_OUI = int(df['desir_bin'].sum())
N_NON = N - N_OUI

print(f"✓ Données chargées : {N:,} femmes")
print(f"  Désire     : {N_OUI:,} ({N_OUI/N*100:.1f}%)")
print(f"  Ne désire  : {N_NON:,} ({N_NON/N*100:.1f}%)")

✓ Données chargées : 13,527 femmes
  Désire     : 12,782 (94.5%)
  Ne désire  : 745 (5.5%)


In [64]:
# ── Visualisation du déséquilibre ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Avant SMOTE
axes[0].bar(['Désire (1)', 'Ne désire pas (0)'], [N_OUI, N_NON],
            color=[BLUE, CORAL], edgecolor='white', linewidth=1.5)
for i, (v, lbl) in enumerate([(N_OUI, f'{N_OUI:,}\n({N_OUI/N*100:.1f}%)'),
                                (N_NON, f'{N_NON:,}\n({N_NON/N*100:.1f}%)')]):
    axes[0].text(i, v + 50, lbl, ha='center', va='bottom', fontsize=10)
axes[0].set_title("Avant SMOTE — Déséquilibre fort", fontsize=11, fontweight='bold')
axes[0].set_ylabel("Nombre d'observations")
axes[0].set_ylim(0, N_OUI * 1.15)

# Après SMOTE (simulation visuelle — les vraies valeurs seront après le split)
axes[1].bar(['Désire (1)', 'Ne désire pas (0)'], [N_OUI, N_OUI],
            color=[BLUE, GREEN], edgecolor='white', linewidth=1.5, alpha=0.8)
axes[1].text(0, N_OUI + 50, f'{N_OUI:,}\n(50%)', ha='center', va='bottom', fontsize=10)
axes[1].text(1, N_OUI + 50, f'~{N_OUI:,}\n(50%) synthétique', ha='center', va='bottom', fontsize=10)
axes[1].set_title("Après SMOTE — Classes équilibrées", fontsize=11, fontweight='bold')
axes[1].set_ylim(0, N_OUI * 1.2)

fig.suptitle("Impact de SMOTE sur la distribution des classes", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("../outputs/figures/fig_smote_distribution.png", bbox_inches='tight')
plt.show()
print("✓ Figure déséquilibre sauvegardée")

✓ Figure déséquilibre sauvegardée


In [65]:
# ── 3.1 Définition de X et y ─────────────────────────────
VARS_MODELE = [
    'age', 'niveau_instruction', 'nb_enfants_vivants',
    'contraceptif_bin', 'statut_matrimonial', 'milieu_residence',
    'quintile_richesse', 'travail', 'region_nord', 'religion_musulman'
]

df_mod = df[VARS_MODELE + ['desir_bin']].dropna().copy()
X      = df_mod[VARS_MODELE].astype(float).values
y      = df_mod['desir_bin'].astype(int).values

print(f"X (variables explicatives) : {X.shape}")
print(f"y (variable cible)         : {y.shape}")
print(f"  Classe 1 (désire)        : {y.sum():,} ({y.mean()*100:.1f}%)")
print(f"  Classe 0 (ne désire pas) : {(y==0).sum():,} ({(y==0).mean()*100:.1f}%)")

X (variables explicatives) : (13527, 10)
y (variable cible)         : (13527,)
  Classe 1 (désire)        : 12,782 (94.5%)
  Classe 0 (ne désire pas) : 745 (5.5%)


In [66]:
# ── 3.2 Split 80/20 STRATIFIÉ ────────────────────────────
# Stratifié = on conserve les proportions 94,5%/5,5% dans train ET test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Split 80/20 stratifié :")
print(f"  Entraînement : {len(X_train):,} obs. → Classe 1: {y_train.sum():,} ({y_train.mean()*100:.1f}%)")
print(f"  Test         : {len(X_test):,} obs.  → Classe 1: {y_test.sum():,} ({y_test.mean()*100:.1f}%)")
print("\n✓ Les proportions sont bien conservées dans les deux parties")

Split 80/20 stratifié :
  Entraînement : 10,821 obs. → Classe 1: 10,225 (94.5%)
  Test         : 2,706 obs.  → Classe 1: 2,557 (94.5%)

✓ Les proportions sont bien conservées dans les deux parties


In [67]:
# ── 3.3 Application de SMOTE sur le TRAIN uniquement ─────
# ⚠ JAMAIS sur le test — sinon on évalue sur des données synthétiques
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("Données d'entraînement APRÈS SMOTE :")
print(f"  Total       : {len(X_train_sm):,} observations")
print(f"  Classe 1    : {y_train_sm.sum():,} ({y_train_sm.mean()*100:.1f}%)")
print(f"  Classe 0    : {(y_train_sm==0).sum():,} ({(y_train_sm==0).mean()*100:.1f}%)")
print(f"\n  Observations créées par SMOTE : {len(X_train_sm)-len(X_train):,}")
print("\nDonnées de test (INCHANGÉES) :")
print(f"  Total       : {len(X_test):,} observations")
print(f"  Classe 1    : {y_test.sum():,} ({y_test.mean()*100:.1f}%)")

Données d'entraînement APRÈS SMOTE :
  Total       : 20,450 observations
  Classe 1    : 10,225 (50.0%)
  Classe 0    : 10,225 (50.0%)

  Observations créées par SMOTE : 9,629

Données de test (INCHANGÉES) :
  Total       : 2,706 observations
  Classe 1    : 2,557 (94.5%)


In [68]:
# ── 3.4 Normalisation pour SVM et KNN ────────────────────
# SVM et KNN sont sensibles à l'échelle des variables
# (une variable de 0-50 dominerait une variable de 0-1)
# On normalise : moyenne=0, écart-type=1
scaler       = StandardScaler()
X_train_sc   = scaler.fit_transform(X_train_sm)   # Ajuster sur le train
X_test_sc    = scaler.transform(X_test)            # Appliquer sur le test

print("✓ Normalisation (StandardScaler) appliquée")
print("  → Utilisée pour SVM et KNN uniquement")
print("  → RL, RF, Arbre, XGBoost utilisent les données brutes")

✓ Normalisation (StandardScaler) appliquée
  → Utilisée pour SVM et KNN uniquement
  → RL, RF, Arbre, XGBoost utilisent les données brutes


In [69]:
# ── Définition des 6 modèles ─────────────────────────────
modeles = {
    'Régression Logistique': LogisticRegression(
        max_iter=1000, random_state=42, class_weight='balanced'),

    'Random Forest': RandomForestClassifier(
        n_estimators=100, random_state=42, n_jobs=-1),

    'Arbre de Décision': DecisionTreeClassifier(
        max_depth=5, random_state=42),

    'SVM': SVC(
        kernel='rbf', probability=True, random_state=42),

    'KNN (k=5)': KNeighborsClassifier(
        n_neighbors=5, n_jobs=-1),

    'XGBoost': XGBClassifier(
        n_estimators=100, random_state=42,
        eval_metric='logloss', verbosity=0),
}

print("6 modèles définis :")
for nom in modeles:
    print(f"  ✓ {nom}")

6 modèles définis :
  ✓ Régression Logistique
  ✓ Random Forest
  ✓ Arbre de Décision
  ✓ SVM
  ✓ KNN (k=5)
  ✓ XGBoost


In [70]:
# ── Entraînement et évaluation des 6 modèles ────────────
resultats = []
roc_data  = {}

print(f"{'Modèle':<25} {'Acc.':>8} {'AUC':>8} {'F1':>8} {'Précision':>10} {'Rappel':>8}")
print("-" * 70)

for nom, clf in modeles.items():
    # Choisir les données normalisées ou brutes
    if nom in ['SVM', 'KNN (k=5)']:
        Xtr, Xte = X_train_sc, X_test_sc
    else:
        Xtr, Xte = X_train_sm, X_test

    # Entraînement
    clf.fit(Xtr, y_train_sm)

    # Prédictions sur le TEST
    y_pred  = clf.predict(Xte)
    y_proba = clf.predict_proba(Xte)[:, 1]

    # Métriques sur le TEST
    acc   = accuracy_score(y_test, y_pred)
    auc   = roc_auc_score(y_test, y_proba)
    f1    = f1_score(y_test, y_pred)
    prec  = precision_score(y_test, y_pred)
    rec   = recall_score(y_test, y_pred)

    print(f"{nom:<25} {acc:>8.4f} {auc:>8.4f} {f1:>8.4f} {prec:>10.4f} {rec:>8.4f}")

    # Courbe ROC
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_data[nom] = {'fpr': fpr, 'tpr': tpr, 'auc': auc}

    resultats.append({
        'Modèle': nom, 'Accuracy': round(acc,4), 'AUC': round(auc,4),
        'F1': round(f1,4), 'Précision': round(prec,4), 'Rappel': round(rec,4)
    })

print("\n✓ Entraînement et évaluation terminés")

Modèle                        Acc.      AUC       F1  Précision   Rappel
----------------------------------------------------------------------
Régression Logistique       0.7738   0.8077   0.8663     0.9817   0.7751
Random Forest               0.9298   0.7453   0.9633     0.9512   0.9758
Arbre de Décision           0.8758   0.7850   0.9317     0.9700   0.8964
SVM                         0.8914   0.8081   0.9409     0.9685   0.9147
KNN (k=5)                   0.8788   0.6703   0.9341     0.9600   0.9097
XGBoost                     0.9350   0.7725   0.9662     0.9504   0.9824

✓ Entraînement et évaluation terminés


In [71]:
# ── Validation croisée 5-fold ────────────────────────────
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"{'Modèle':<25} {'AUC CV-5':>10} {'Écart-type':>12}")
print("-" * 50)

cv_resultats = {}
for nom, clf in modeles.items():
    if nom in ['SVM', 'KNN (k=5)']:
        Xtr = X_train_sc
    else:
        Xtr = X_train_sm

    cv_scores = cross_val_score(clf, Xtr, y_train_sm,
                                cv=skf, scoring='roc_auc', n_jobs=-1)
    auc_cv  = cv_scores.mean()
    auc_std = cv_scores.std()
    print(f"{nom:<25} {auc_cv:>10.4f} {auc_std:>12.4f}")
    cv_resultats[nom] = {'AUC_CV5': round(auc_cv,4), 'Écart-type': round(auc_std,4)}

print("\n✓ Validation croisée terminée")
print("  → Plus l'AUC CV-5 est élevé et l'écart-type faible, meilleur est le modèle")

Modèle                      AUC CV-5   Écart-type
--------------------------------------------------
Régression Logistique         0.8098       0.0058
Random Forest                 0.9910       0.0014
Arbre de Décision             0.8956       0.0070
SVM                           0.9522       0.0017
KNN (k=5)                     0.9714       0.0011
XGBoost                       0.9881       0.0016

✓ Validation croisée terminée
  → Plus l'AUC CV-5 est élevé et l'écart-type faible, meilleur est le modèle


In [72]:
# ── Calcul du gap train/test pour chaque modèle ──────────
print(f"{'Modèle':<25} {'AUC Train':>10} {'AUC Test':>10} {'Gap':>10}   Diagnostic")
print("-" * 72)

overfitting_data = []
for nom, clf in modeles.items():
    if nom in ['SVM', 'KNN (k=5)']:
        Xtr, Xte = X_train_sc, X_test_sc
    else:
        Xtr, Xte = X_train_sm, X_test

    # AUC sur les données d'entraînement
    y_train_proba = clf.predict_proba(Xtr)[:, 1]
    auc_train     = roc_auc_score(y_train_sm, y_train_proba)

    # AUC sur les données de test
    auc_test      = roc_data[nom]['auc']

    gap  = auc_train - auc_test
    diag = (" Bon ajustement" if gap <= 0.05
            else ("Léger overfitting" if gap <= 0.15
            else "Overfitting sévère"))

    print(f"{nom:<25} {auc_train:>10.4f} {auc_test:>10.4f} {gap:>10.4f}   {diag}")
    overfitting_data.append({
        'Modèle': nom, 'AUC_train': round(auc_train,4),
        'AUC_test': auc_test, 'Gap': round(gap,4), 'Diagnostic': diag
    })

ov_df = pd.DataFrame(overfitting_data)
print("\n→ Un gap élevé signifie que le modèle a sur-appris les données d'entraînement")

Modèle                     AUC Train   AUC Test        Gap   Diagnostic
------------------------------------------------------------------------
Régression Logistique         0.8102     0.8077     0.0025    Bon ajustement
Random Forest                 0.9996     0.7453     0.2543   Overfitting sévère
Arbre de Décision             0.9004     0.7850     0.1154   Léger overfitting
SVM                           0.9580     0.8081     0.1498   Léger overfitting
KNN (k=5)                     0.9939     0.6703     0.3236   Overfitting sévère
XGBoost                       0.9966     0.7725     0.2241   Overfitting sévère

→ Un gap élevé signifie que le modèle a sur-appris les données d'entraînement


In [73]:
# ── Figure : Gap overfitting ─────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
x      = np.arange(len(ov_df))
width  = 0.35
noms   = ov_df['Modèle'].tolist()

bars1 = ax.bar(x - width/2, ov_df['AUC_train'], width,
               label='AUC (entraînement)', color=BLUE, alpha=0.85, edgecolor='white')
bars2 = ax.bar(x + width/2, ov_df['AUC_test'], width,
               label='AUC (test)', color=CORAL, alpha=0.85, edgecolor='white')

for b in list(bars1) + list(bars2):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.005,
            f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=8.5)

ax.set_xticks(x)
ax.set_xticklabels(noms, rotation=10, ha='right', fontsize=9)
ax.set_ylim(0, 1.15)
ax.set_ylabel("AUC-ROC", fontsize=11)
ax.set_title("Analyse de l'overfitting — AUC Train vs AUC Test\n"
             "(un écart important = overfitting)", fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.axhline(y=0.75, color=GRAY, linestyle='--', alpha=0.5, label='Seuil acceptable (0.75)')

plt.tight_layout()
plt.savefig("../outputs/figures/fig_overfitting_analysis.png", bbox_inches='tight')
plt.show()
print("✓ Figure overfitting sauvegardée")

✓ Figure overfitting sauvegardée


In [74]:
# ── Tableau de synthèse complet ──────────────────────────
# Fusionner toutes les métriques
res_df = pd.DataFrame(resultats)
cv_df  = pd.DataFrame([
    {'Modèle': nom, **vals} for nom, vals in cv_resultats.items()
])
ov_df2 = ov_df[['Modèle','AUC_train','Gap','Diagnostic']]

synthese = (res_df
    .merge(cv_df,  on='Modèle')
    .merge(ov_df2, on='Modèle')
    .sort_values('AUC_CV5', ascending=False)
    .reset_index(drop=True))

# Affichage
print("=" * 90)
print("TABLEAU DE COMPARAISON COMPLET DES 6 MODÈLES ML")
print("=" * 90)
cols_affich = ['Modèle','Accuracy','AUC','F1','AUC_CV5','Gap','Diagnostic']
print(synthese[cols_affich].to_string(index=False))

synthese.to_csv("../outputs/tables/06_comparaison_modeles_ML.csv", index=False)
print("\n✓ Sauvegardé → 06_comparaison_modeles_ML.csv")

TABLEAU DE COMPARAISON COMPLET DES 6 MODÈLES ML
               Modèle  Accuracy    AUC     F1  AUC_CV5    Gap         Diagnostic
        Random Forest    0.9298 0.7453 0.9633   0.9910 0.2543 Overfitting sévère
              XGBoost    0.9350 0.7725 0.9662   0.9881 0.2241 Overfitting sévère
            KNN (k=5)    0.8788 0.6703 0.9341   0.9714 0.3236 Overfitting sévère
                  SVM    0.8914 0.8081 0.9409   0.9522 0.1498  Léger overfitting
    Arbre de Décision    0.8758 0.7850 0.9317   0.8956 0.1154  Léger overfitting
Régression Logistique    0.7738 0.8077 0.8663   0.8098 0.0025     Bon ajustement

✓ Sauvegardé → 06_comparaison_modeles_ML.csv


In [75]:
# ── Figure : Comparaison des performances ────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gauche : Accuracy + AUC par modèle
x     = np.arange(len(synthese))
width = 0.35
b1 = axes[0].bar(x-width/2, synthese['Accuracy'], width,
                 label='Accuracy', color=NAVY, alpha=0.85, edgecolor='white')
b2 = axes[0].bar(x+width/2, synthese['AUC'], width,
                 label='AUC-ROC', color=CORAL, alpha=0.85, edgecolor='white')
for b in list(b1)+list(b2):
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+0.005,
                 f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(synthese['Modèle'], rotation=15, ha='right', fontsize=8.5)
axes[0].set_ylim(0, 1.12)
axes[0].set_ylabel("Score")
axes[0].set_title("Accuracy vs AUC-ROC", fontsize=11, fontweight='bold')
axes[0].legend(fontsize=9)

# Droite : AUC-CV5 + F1
b3 = axes[1].bar(x-width/2, synthese['AUC_CV5'], width,
                 label='AUC CV-5', color=TEAL, alpha=0.85, edgecolor='white')
b4 = axes[1].bar(x+width/2, synthese['F1'], width,
                 label='F1-score', color=AMBER, alpha=0.85, edgecolor='white')
for b in list(b3)+list(b4):
    axes[1].text(b.get_x()+b.get_width()/2, b.get_height()+0.005,
                 f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(synthese['Modèle'], rotation=15, ha='right', fontsize=8.5)
axes[1].set_ylim(0, 1.12)
axes[1].set_ylabel("Score")
axes[1].set_title("AUC CV-5 vs F1-score", fontsize=11, fontweight='bold')
axes[1].legend(fontsize=9)

fig.suptitle("Comparaison des 6 modèles ML (après SMOTE)",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("../outputs/figures/fig10_comparaison_modeles.png", bbox_inches='tight')
plt.show()
print("✓ Figure comparaison sauvegardée")

✓ Figure comparaison sauvegardée


In [76]:
# ── Figure : Courbes ROC superposées ─────────────────────
# La courbe ROC trace sensibilité vs (1 - spécificité) pour tous les seuils
# Plus la courbe est "bombée" vers le coin supérieur gauche, meilleur est le modèle
fig, ax = plt.subplots(figsize=(8, 7))

for (nom, data), color in zip(roc_data.items(), COLORS_6):
    ax.plot(data['fpr'], data['tpr'],
            label=f"{nom} (AUC={data['auc']:.3f})",
            color=color, linewidth=2.2)

ax.plot([0,1], [0,1], 'k--', linewidth=1, alpha=0.5, label='Aléatoire (AUC=0.500)')
ax.fill_between([0,1], [0,1], alpha=0.05, color='gray')
ax.set_xlabel("Taux de faux positifs (1 − Spécificité)", fontsize=11)
ax.set_ylabel("Taux de vrais positifs (Sensibilité)", fontsize=11)
ax.set_title("Courbes ROC — Comparaison des 6 modèles ML\n(après SMOTE)",
             fontsize=12, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig("../outputs/figures/fig8_roc_curves.png", bbox_inches='tight')
plt.show()
print("✓ Courbes ROC sauvegardées")

✓ Courbes ROC sauvegardées


In [77]:
# ── Figure : Importance des variables (Random Forest) ─────
# Le Random Forest donne une mesure d'importance pour chaque variable
# = combien chaque variable réduit l'impureté (Gini) dans les arbres

rf = modeles['Random Forest']
imp = pd.Series(rf.feature_importances_, index=VARS_MODELE).sort_values(ascending=True)

labels_imp = {
    'age':'Âge', 'niveau_instruction':"Niveau d'instruction",
    'nb_enfants_vivants':'Nb enfants vivants', 'contraceptif_bin':'Contraceptif',
    'statut_matrimonial':'Statut matrimonial', 'milieu_residence':'Milieu résidence',
    'quintile_richesse':'Quintile richesse', 'travail':'Emploi (travail)',
    'region_nord':'Région septentrionale', 'religion_musulman':'Religion musulmane',
}
imp.index = [labels_imp.get(i,i) for i in imp.index]

fig, ax = plt.subplots(figsize=(9, 5.5))
colors_imp = [NAVY if v >= imp.median() else BLUE for v in imp.values]
bars = ax.barh(imp.index, imp.values, color=colors_imp, edgecolor='white', linewidth=1.5)
for b in bars:
    ax.text(b.get_width()+0.003, b.get_y()+b.get_height()/2,
            f'{b.get_width():.3f} ({b.get_width()*100:.1f}%)',
            va='center', fontsize=9.5, fontweight='bold')
ax.set_xlabel("Importance (impureté de Gini)", fontsize=11)
ax.set_title("Importance des variables — Random Forest\n(après SMOTE)",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("../outputs/figures/fig9_importance_variables.png", bbox_inches='tight')
plt.show()

imp.sort_values(ascending=False).to_csv(
    "../outputs/tables/07_importance_variables.csv", header=['Importance'])
print("✓ Importance des variables sauvegardée")

✓ Importance des variables sauvegardée


In [78]:
# ── Sélection automatique du meilleur modèle ─────────────
# Critère : AUC CV-5 le plus élevé parmi les modèles sans overfitting sévère
candidats = synthese[synthese['Gap'] <= 0.15].copy()  # Exclure overfitting sévère
best_row   = candidats.sort_values('AUC_CV5', ascending=False).iloc[0]
best_nom   = best_row['Modèle']

print("=" * 60)
print("MEILLEUR MODÈLE SÉLECTIONNÉ")
print("=" * 60)
print(f"\n  Modèle    : {best_nom}")
print(f"  Accuracy  : {best_row['Accuracy']:.4f}")
print(f"  AUC test  : {best_row['AUC']:.4f}")
print(f"  AUC CV-5  : {best_row['AUC_CV5']:.4f}")
print(f"  F1-score  : {best_row['F1']:.4f}")
print(f"  Gap       : {best_row['Gap']:.4f}  ({best_row['Diagnostic']})")

# Justification
print("\n  Justification :")
print(f"  → AUC CV-5 = {best_row['AUC_CV5']:.4f} : meilleure estimation robuste")
print(f"  → Gap = {best_row['Gap']:.4f} : bonne généralisation (pas d'overfitting)")
print(f"  → AUC > 0.75 : seuil satisfaisant pour données EDS")

MEILLEUR MODÈLE SÉLECTIONNÉ

  Modèle    : SVM
  Accuracy  : 0.8914
  AUC test  : 0.8081
  AUC CV-5  : 0.9522
  F1-score  : 0.9409
  Gap       : 0.1498  (Léger overfitting)

  Justification :
  → AUC CV-5 = 0.9522 : meilleure estimation robuste
  → Gap = 0.1498 : bonne généralisation (pas d'overfitting)
  → AUC > 0.75 : seuil satisfaisant pour données EDS


In [79]:
# ── Matrice de confusion du meilleur modèle ──────────────
best_clf = modeles[best_nom]
if best_nom in ['SVM', 'KNN (k=5)']:
    y_pred_best  = best_clf.predict(X_test_sc)
    y_proba_best = best_clf.predict_proba(X_test_sc)[:, 1]
else:
    y_pred_best  = best_clf.predict(X_test)
    y_proba_best = best_clf.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_pred_best)
tn, fp, fn, tp = cm.ravel()
sensib = tp / (tp+fn) * 100
specif = tn / (tn+fp) * 100
pct_ok = (tp+tn) / len(y_test) * 100

print(f"Matrice de confusion — {best_nom} :")
print(f"  VP = {tp:,}  |  VN = {tn:,}")
print(f"  FP = {fp:,}  |  FN = {fn:,}")
print(f"  Sensibilité  : {sensib:.1f}%")
print(f"  Spécificité  : {specif:.1f}%")
print(f"  % bien classés: {pct_ok:.1f}%")
print(f"\nReport complet :")
print(classification_report(y_test, y_pred_best,
      target_names=['Ne désire pas (0)','Désire (1)']))

Matrice de confusion — SVM :
  VP = 2,339  |  VN = 73
  FP = 76  |  FN = 218
  Sensibilité  : 91.5%
  Spécificité  : 49.0%
  % bien classés: 89.1%

Report complet :
                   precision    recall  f1-score   support

Ne désire pas (0)       0.25      0.49      0.33       149
       Désire (1)       0.97      0.91      0.94      2557

         accuracy                           0.89      2706
        macro avg       0.61      0.70      0.64      2706
     weighted avg       0.93      0.89      0.91      2706



In [80]:
# ── Matrice de confusion — Figure ────────────────────────
fig, ax = plt.subplots(figsize=(6.5, 5.5))
cm_labels = np.array([
    [f'VN = {tn:,}\n(Spécificité)', f'FP = {fp:,}'],
    [f'FN = {fn:,}',                  f'VP = {tp:,}\n(Sensibilité)']
])
im = ax.imshow([[tn,fp],[fn,tp]], interpolation='nearest', cmap='Blues')
ax.set_xticks([0,1])
ax.set_yticks([0,1])
ax.set_xticklabels(['Prédit : Ne désire pas','Prédit : Désire'], fontsize=9)
ax.set_yticklabels(['Réel : Ne désire pas','Réel : Désire'], fontsize=9)
for i in range(2):
    for j in range(2):
        val = [[tn,fp],[fn,tp]][i][j]
        ax.text(j, i, cm_labels[i,j], ha='center', va='center', fontsize=10,
                color='white' if val > max(tn,fp,fn,tp)/2 else 'black')
ax.set_title(f"Matrice de confusion — {best_nom}\n"
             f"% bien classés : {pct_ok:.1f}%  |  "
             f"Sensib. : {sensib:.1f}%  |  Spécif. : {specif:.1f}%",
             fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig("../outputs/figures/figD_matrice_confusion.png", bbox_inches='tight')
plt.show()
print(f"✓ Matrice de confusion ({best_nom}) sauvegardée")

✓ Matrice de confusion (SVM) sauvegardée


In [81]:
# ── Résumé final ─────────────────────────────────────────
print("=" * 60)
print("ANALYSE ML TERMINÉE — RÉSUMÉ")
print("=" * 60)
print(f"\nN = {N:,} femmes · SMOTE appliqué · 6 modèles comparés")
print(f"\nClassement par AUC CV-5 :")
for i, row in synthese[['Modèle','AUC','AUC_CV5','Gap','Diagnostic']].iterrows():
    flag = "← RETENU" if row['Modèle'] == best_nom else ""
    print(f"  {i+1}. {row['Modèle']:<25} AUC={row['AUC']:.3f} · CV5={row['AUC_CV5']:.3f} · Gap={row['Gap']:.3f}  {flag}")

print(f"\n🏆 Meilleur modèle : {best_nom}")
print(f"   AUC CV-5 = {best_row['AUC_CV5']:.4f}  ·  Gap = {best_row['Gap']:.4f}")
print(f"\nFichiers générés :")
for f in sorted(os.listdir("../outputs/tables/")):
    if f.startswith("06") or f.startswith("07"):
        print(f"  📊 {f}")
for f in sorted(os.listdir("../outputs/figures/")):
    if any(x in f for x in ['smote','overfitting','comparaison','roc','importance','confusion']):
        print(f"  🖼 {f}")

ANALYSE ML TERMINÉE — RÉSUMÉ

N = 13,527 femmes · SMOTE appliqué · 6 modèles comparés

Classement par AUC CV-5 :
  1. Random Forest             AUC=0.745 · CV5=0.991 · Gap=0.254  
  2. XGBoost                   AUC=0.772 · CV5=0.988 · Gap=0.224  
  3. KNN (k=5)                 AUC=0.670 · CV5=0.971 · Gap=0.324  
  4. SVM                       AUC=0.808 · CV5=0.952 · Gap=0.150  ← RETENU
  5. Arbre de Décision         AUC=0.785 · CV5=0.896 · Gap=0.115  
  6. Régression Logistique     AUC=0.808 · CV5=0.810 · Gap=0.003  

🏆 Meilleur modèle : SVM
   AUC CV-5 = 0.9522  ·  Gap = 0.1498

Fichiers générés :
  📊 06_comparaison_modeles_ML.csv
  📊 07_importance_variables.csv
  🖼 fig10_comparaison_modeles.png
  🖼 fig8_roc_curves.png
  🖼 fig9_importance_variables.png
  🖼 figD_matrice_confusion.png
  🖼 fig_overfitting_analysis.png
  🖼 fig_shap_importance.png
  🖼 fig_smote_distribution.png


In [82]:
# ── Installation et import SHAP ─────────────────────────────
import shap
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── Noms lisibles des variables ──────────────────────────────
LABELS_SHAP = {
    'age'                : 'Âge',
    'niveau_instruction' : "Niveau d'instruction",
    'nb_enfants_vivants' : 'Nb enfants vivants',
    'contraceptif_bin'   : 'Utilisation contraceptif',
    'statut_matrimonial' : 'Statut matrimonial',
    'milieu_residence'   : 'Milieu de résidence',
    'quintile_richesse'  : 'Quintile de richesse',
    'travail'            : 'Emploi (travail)',
    'region_nord'        : 'Région septentrionale',
    'religion_musulman'  : 'Religion musulmane',
}

# ── Créer un DataFrame X_test avec les noms de colonnes ─────
X_test_df = pd.DataFrame(X_test, columns=VARS_MODELE)
X_test_df.columns = [LABELS_SHAP.get(c, c) for c in X_test_df.columns]

# ── Explainer SHAP pour la Régression Logistique ─────────────
# LinearExplainer est optimal pour les modèles linéaires
best_clf_rl = modeles['Régression Logistique']
explainer = shap.LinearExplainer(best_clf_rl, X_test, feature_names=list(X_test_df.columns))
shap_values = explainer(X_test)

print(f"✓ SHAP calculé pour {len(X_test):,} observations")
print(f"  Shape des SHAP values : {shap_values.values.shape}")
print(f"  Variables : {list(X_test_df.columns)}")

Background dataset has 2706 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=2706 when initializing the masker.


✓ SHAP calculé pour 2,706 observations
  Shape des SHAP values : (2706, 10)
  Variables : ['Âge', "Niveau d'instruction", 'Nb enfants vivants', 'Utilisation contraceptif', 'Statut matrimonial', 'Milieu de résidence', 'Quintile de richesse', 'Emploi (travail)', 'Région septentrionale', 'Religion musulmane']


In [83]:
# ── Figure SHAP 1 : Bar plot (importance globale) ───────────
fig, ax = plt.subplots(figsize=(9, 5.5))

# Calcul de l'importance moyenne absolue
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
feature_names = list(X_test_df.columns)
importance_df = pd.DataFrame({
    'Variable': feature_names,
    'Importance SHAP': mean_abs_shap
}).sort_values('Importance SHAP', ascending=True)

# Couleurs selon l'importance
colors = ['#0D1B4B' if v >= importance_df['Importance SHAP'].median() else '#378ADD'
          for v in importance_df['Importance SHAP']]

bars = ax.barh(importance_df['Variable'], importance_df['Importance SHAP'],
               color=colors, edgecolor='white', linewidth=1.2)
for b in bars:
    ax.text(b.get_width() + 0.001, b.get_y() + b.get_height()/2,
            f'{b.get_width():.4f}',
            va='center', fontsize=9.5, fontweight='bold')

ax.set_xlabel("Importance SHAP moyenne (|valeur|)", fontsize=11)
ax.set_title("Importance des variables — Analyse SHAP\n"
             "Régression Logistique (EDSC Cameroun 2018)",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("../outputs/figures/fig_shap_importance.png", bbox_inches='tight')
plt.show()
print("✓ SHAP Bar plot sauvegardé → fig_shap_importance.png")

✓ SHAP Bar plot sauvegardé → fig_shap_importance.png


In [84]:
# ── Figure SHAP 2 : Beeswarm plot ───────────────────────────
plt.figure(figsize=(10, 6))
shap.plots.beeswarm(shap_values, max_display=10, show=False)
plt.title("SHAP Beeswarm — Distribution des contributions par variable\n"
          "Rouge=valeur élevée · Bleu=valeur faible · Axe horizontal=impact sur la prédiction",
          fontsize=11, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig("../outputs/figures/fig_shap_beeswarm.png", bbox_inches='tight')
plt.show()
print("✓ SHAP Beeswarm sauvegardé → fig_shap_beeswarm.png")
print("\nLecture :")
print("  Points rouges à gauche (âge) → âge élevé = moins de chance de désirer")
print("  Points rouges à droite (contraceptif) → utilisation = plus de chance de désirer")

✓ SHAP Beeswarm sauvegardé → fig_shap_beeswarm.png

Lecture :
  Points rouges à gauche (âge) → âge élevé = moins de chance de désirer
  Points rouges à droite (contraceptif) → utilisation = plus de chance de désirer


In [85]:
# ── Figure SHAP 3 : Waterfall pour une femme "ne désire pas" ─
# Trouver une femme de la classe 0 bien classée
y_pred_best = modeles['Régression Logistique'].predict(X_test)
idx_bien_classes_0 = np.where((y_test == 0) & (y_pred_best == 0))[0]

if len(idx_bien_classes_0) > 0:
    idx = idx_bien_classes_0[0]  # première femme bien classée dans la classe 0
    plt.figure(figsize=(10, 5.5))
    shap.plots.waterfall(shap_values[idx], show=False)
    plt.title(f"SHAP Waterfall — Femme n°{idx} (prédiction correcte : Ne désire pas)\n"
              "Décomposition de la prédiction individuelle",
              fontsize=11, fontweight='bold', pad=15)
    plt.tight_layout()
    plt.savefig("../outputs/figures/fig_shap_waterfall.png", bbox_inches='tight')
    plt.show()
    print(f"✓ SHAP Waterfall sauvegardé → fig_shap_waterfall.png")
    print(f"  Femme sélectionnée : observation n°{idx}")
    print(f"  Âge : {X_test[idx, 0]:.0f} ans")
    print(f"  Nb enfants vivants : {X_test[idx, 2]:.0f}")
else:
    print("Aucune observation bien classée dans la classe 0 — sélection alternative")

✓ SHAP Waterfall sauvegardé → fig_shap_waterfall.png
  Femme sélectionnée : observation n°12
  Âge : 45 ans
  Nb enfants vivants : 7


In [ ]:
# ── Résumé de l'analyse SHAP ────────────────────────────────
print("=" * 60)
print("RÉSUMÉ SHAP — PRINCIPAUX ENSEIGNEMENTS")
print("=" * 60)

for v, s in zip(importance_df.sort_values('Importance SHAP', ascending=False)['Variable'],
                importance_df.sort_values('Importance SHAP', ascending=False)['Importance SHAP']):
    # feature_names contient déjà les labels français — recherche directe
    col_idx   = feature_names.index(v)
    direction = shap_values.values[:, col_idx].mean()
    sens = "→ réduit le désir" if direction < 0 else "→ augmente le désir"
    print(f"  {v:<28} SHAP={s:.4f}  {sens}")

print("\n✓ Fichiers SHAP sauvegardés :")
for f in ['fig_shap_importance.png', 'fig_shap_beeswarm.png', 'fig_shap_waterfall.png']:
    print(f"  🖼 outputs/figures/{f}")

In [ ]:
# ── Permutation Importance — SVM calibré (modèle sélectionné) ──────────
# Pour le SVM calibré, SHAP LinearExplainer ne s'applique pas.
# Permutation Importance : on perturbe chaque variable aléatoirement
# et on mesure la chute d'AUC → variable importante = grande chute.
from sklearn.inspection import permutation_importance

LABELS_PI = {
    'age': 'Age',
    'niveau_instruction': 'Niveau instruction',
    'nb_enfants_vivants': 'Nb enfants vivants',
    'contraceptif_bin': 'Utilisation contraceptif',
    'statut_matrimonial': 'Statut matrimonial',
    'milieu_residence': 'Milieu de residence',
    'quintile_richesse': 'Quintile de richesse',
    'travail': 'Emploi (travail)',
    'region_nord': 'Region septentrionale',
    'religion_musulman': 'Religion musulmane',
}

best_clf_svm = modeles['SVM']
r_perm = permutation_importance(
    best_clf_svm, X_test_sc, y_test,
    n_repeats=10, random_state=42, scoring='roc_auc', n_jobs=-1
)

perm_df = pd.DataFrame({
    'Variable': [LABELS_PI[v] for v in VARS_MODELE],
    'Importance': r_perm.importances_mean,
    'Ecart_type': r_perm.importances_std,
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print('Permutation Importance --- SVM calibre (chute AUC quand variable permutee) :')
print(f"{'Variable':<30} {'Chute AUC':>12} {'Ecart-type':>12}")
print('-' * 56)
for _, row in perm_df.iterrows():
    print(f"{row['Variable']:<30} {row['Importance']:>+12.4f} {row['Ecart_type']:>12.4f}")


In [ ]:
# ── Figure : Permutation Importance SVM ─────────────────────────────────
perm_plot = perm_df.sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(9, 5.5))
colors_pi = [NAVY if v >= 0 else CORAL for v in perm_plot['Importance']]
bars = ax.barh(
    perm_plot['Variable'], perm_plot['Importance'],
    xerr=perm_plot['Ecart_type'],
    color=colors_pi, edgecolor='white', linewidth=1.2,
    capsize=4, error_kw={'elinewidth': 1.2, 'ecolor': GRAY}
)
ax.axvline(x=0, color='gray', linewidth=1, linestyle='--', alpha=0.7)
for b, v in zip(bars, perm_plot['Importance']):
    offset = 0.002 if v >= 0 else -0.002
    ha = 'left' if v >= 0 else 'right'
    ax.text(b.get_width() + offset, b.get_y() + b.get_height() / 2,
            f'{v:+.4f}', va='center', fontsize=9, fontweight='bold', ha=ha)
ax.set_xlabel(
    "Chute d'AUC-ROC quand la variable est permutee\n"
    "(valeur positive = variable importante ; proche de 0 = peu utile)",
    fontsize=11
)
ax.set_title(
    'Permutation Importance --- SVM calibre (modele selectionne)\n'
    'EDSC Cameroun 2018 - n=2 706 observations de test',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig('../outputs/figures/fig_permutation_importance_svm.png', bbox_inches='tight')
plt.show()
print('Figure sauvegardee : fig_permutation_importance_svm.png')


In [ ]:
# ── Courbe de calibration — SVM calibre vs Regression Logistique ────────
# Si le modele predit 70 %, est-ce que ~70 % des femmes desirent vraiment ?
# Une courbe proche de la diagonale = probabilites predites fiables.
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(7, 6))

# Diagonale de reference
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5,
        label='Calibration parfaite', alpha=0.6)

# SVM calibre
frac_svm, pred_svm = calibration_curve(
    y_test, y_proba_best, n_bins=10, strategy='uniform')
ax.plot(pred_svm, frac_svm, 'o-',
        color=NAVY, linewidth=2.2, markersize=7,
        label=f'SVM calibre (AUC={roc_data["SVM"]["auc"]:.3f})')

# Regression Logistique pour comparaison
y_proba_rl = modeles['Regression Logistique'].predict_proba(X_test)[:, 1] \
    if 'Regression Logistique' in modeles \
    else modeles[list(modeles.keys())[0]].predict_proba(X_test)[:, 1]
y_proba_rl = modeles['Régression Logistique'].predict_proba(X_test)[:, 1]
frac_rl, pred_rl = calibration_curve(
    y_test, y_proba_rl, n_bins=10, strategy='uniform')
ax.plot(pred_rl, frac_rl, 's--',
        color=CORAL, linewidth=1.8, markersize=6, alpha=0.8,
        label=f'Regression Logistique (AUC={roc_data["Régression Logistique"]["auc"]:.3f})')

ax.set_xlabel('Probabilite predite moyenne', fontsize=11)
ax.set_ylabel('Fraction de cas positifs observes', fontsize=11)
ax.set_title(
    'Courbe de calibration --- SVM calibre vs Regression Logistique\n'
    '(proche de la diagonale = probabilites predites fiables)',
    fontsize=11, fontweight='bold'
)
ax.legend(fontsize=10)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('../outputs/figures/fig_calibration_curve.png', bbox_inches='tight')
plt.show()
print('Figure sauvegardee : fig_calibration_curve.png')
print()
print('Interpretation :')
print('  -> Courbe au-dessus de la diagonale : le modele SOUS-estime la probabilite')
print('  -> Courbe en dessous : le modele SURESTIMÉ la probabilite')
print('  -> Plus pres de la diagonale = probabilites plus fiables')


In [ ]:
# ── Synthese finale pour le memoire ─────────────────────────────────────
print('=' * 68)
print('SYNTHESE COMPLETE --- ANALYSE ML · EDSC CAMEROUN 2018')
print('=' * 68)
print()
print('DONNEES')
print(f'  N = {N:,} femmes de 15-49 ans (EDSC-V 2018, INS Cameroun)')
print(f'  Variable cible : desirer un enfant supplementaire')
print(f'  Prevalence : {N_OUI/N*100:.1f}% desirent / {N_NON/N*100:.1f}% ne desirent pas')
print(f'  Desequilibre corrige par SMOTE (train uniquement)')
print()
print('PIPELINE COMPLET')
print('  1. Import (pyreadstat, CMIR71FL.SAV)')
print('  2. Traitement (encodage, variables derivees, dropna)')
print('  3. Separation X / y (10 variables explicatives / desir_bin)')
print('  4. Split 80/20 stratifie')
print('  5. SMOTE sur le train uniquement (jamais sur le test)')
print('  6. Normalisation StandardScaler (pour SVM)')
print('  7. Validation croisee 5-fold (StratifiedKFold)')
print('  8. Evaluation : AUC, Accuracy, F1, gap overfitting')
print('  9. Selection : meilleur AUC CV-5 parmi les modeles avec gap <= 0.15')
print(' 10. Interpretation : Permutation Importance + courbe de calibration')
print()
print('RESULTATS --- 6 MODELES COMPARES')
print('-' * 68)
for _, row in synthese[['Modèle', 'AUC_CV5', 'AUC', 'Gap', 'Diagnostic']].iterrows():
    flag = ' <-- RETENU' if row['Modèle'] == best_nom else ''
    print(f"  {row['Modèle']:<25} CV5={row['AUC_CV5']:.4f}  Test={row['AUC']:.4f}  Gap={row['Gap']:.4f}{flag}")
print()
print(f'MODELE RETENU : {best_nom}')
print(f'  AUC CV-5     = {best_row["AUC_CV5"]:.4f}  (robustesse sur 5 folds)')
print(f'  AUC test     = {best_row["AUC"]:.4f}  (generalisation sur donnees inedites)')
print(f'  Sensibilite  = {sensib:.1f}%  (femmes qui desirent, bien detectees)')
print(f'  Specificite  = {specif:.1f}%  (classe minoritaire difficile)')
print(f'  % bien classes = {pct_ok:.1f}%')
print()
print('FIGURES GENEREES POUR LE MEMOIRE')
for fig_file in sorted(os.listdir('../outputs/figures/')):
    print(f'  {fig_file}')
print()
print('Pipeline complet termine --- tous les fichiers sont dans outputs/')
